Not a finished project. just a test to see how much nicer using classes are compared to single functions.

In [ ]:
import numpy as np
import sympy as sp
from latex2sympy2 import latex2sympy
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from matplotlib.colors import to_rgb
from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import HTML

In [ ]:
# function to change colour brightness
def rebrighten(colour, value):
    r, g, b = to_rgb(colour)
    return (r * value, g * value, b * value)

In [ ]:
class RiemannSum:
    
    def __init__(
        self, func1, func2=r"0*x", lower=0, upper=1, func_samples=1000
    ):
        """"
        __init__ function takes in function arguments written with LaTeX 
        formatting and the lower and upper bounds to sum between. Supports 
        summing between two different functions, not just with the x-axis.
        """
        
        # calculate sample domain
        self.lower = lower
        self.upper = upper
        x_padding = 0.2 * abs(self.upper - self.lower)
        self.x_vals = np.linspace(
            self.lower - x_padding, self.upper + x_padding, func_samples
        )

        # process function arguments, defaulting to the x-axis
        x_axis = [0] * func_samples
        self.functions = [x_axis.copy(), x_axis.copy()]

        # convert function from LaTeX to SymPy formatting
        self.lambda_funcs = [lambda _: 0, lambda _: 0]
        x = sp.symbols('x')
        for i, func in enumerate([func1, func2]):
            expr = latex2sympy(func)
            self.lambda_funcs[i] = sp.lambdify(x, expr, modules="numpy")
            self.functions[i] = self.lambda_funcs[i](self.x_vals)

        # initialise plot
        plt.style.use("dark_background")
        self.fig, self.ax = plt.subplots(figsize=[16, 8])
        self.ax.set_xlim([min(self.x_vals), max(self.x_vals)])

        # plot functions and other visual aids
        self.ax.axhline(0, alpha=0.3, zorder=-1)
        self.ax.axvline(0, alpha=0.3, zorder=-1)
        self.ax.axvline(
            self.lower, linestyle="--", color="yellow", linewidth=2, alpha=0.5
        )
        self.ax.axvline(
            self.upper, linestyle="--", color="yellow", linewidth=2, alpha=0.5
        )
        self.ax.plot(
            self.x_vals, self.functions[0], color="crimson", linewidth=2
        )
        self.ax.plot(
            self.x_vals, self.functions[1], color="orangered", linewidth=2,
            alpha = 0 if func2 == r"0*x" else 1
        )

    def add_strips(self, n_strips, rule=0.5):
        """
        add_strips function performs Riemann sum of n_strips. The sampling rule 
        parameter implements the "left rule" when 0, the "right rule" when 1, 
        and any other sampling point between 0 and 1. It is defualted to 0.5 to 
        sample from the centre of the strips.
        """

        # calculate with of each strip based on n_strips
        width = (self.upper - self.lower) / n_strips

        # loop over and build each strip
        self.area = 0
        strips = []
        for n in range(1, n_strips + 1):

            # find x value to sample at for both functions
            target = self.lower + (n - 1 + rule) * width

            # find f1(target) and f2(target)
            f1 = self.lambda_funcs[0](target)
            f2 = self.lambda_funcs[1](target)
            
            # calulate strip area
            height = f1 - f2
            single_area = width * height
            self.area += single_area

            # colour strips depending on signed area
            strip_col = rebrighten(
                "midnightblue", 1.2 if single_area < 0 else 0.8
            )

            # build strip
            xy = (self.lower + (n - 1) * width, f2)
            strip = Rectangle(
                xy, width, height, facecolor=strip_col,
                edgecolor="midnightblue", linewidth=0.8
            )
            self.ax.add_patch(strip)
            strips.append(strip)

    def __update__(self, frame):
        """
        __update__ function creates animation frame by frame.
        """

        # clear strips off plot
        for patch in list(self.ax.patches):
            patch.remove()

        # draw on current frame's strips
        if frame > 0: self.add_strips(frame, self.rule)

        return self.ax.patches
    
    def animate(self, start_strips=0, end_strips=100, interval=1, rule=0.5):
        """
        animate function produces a gif of the Riemann sum strips. Starting at
        start_strips to end_strips, with a specified interval of strips to add 
        on per frame.
        """

        # calculate number of frames to animate
        frames = round((end_strips - start_strips) / interval)

        # let frames be the number of strips to display
        frames = list(range(start_strips, end_strips + 1, interval))

        # create animation
        self.rule = rule
        anim = FuncAnimation(self.fig, self.__update__, frames=frames)
        plt.close(self.fig)

        return HTML(anim.to_jshtml())